# 6. Ingreso: calibración del encogimiento y correcciones en log-odds

El notebook 5 alcanzó **0,945159 OOF** frente a 0,942566 de Optuna, y **0,94544 público**. Residual de ingreso: 0,944561; representación/frecuencia: 0,943524; híbrido básico: 0,941985. Movilidad y perfil no mejoraron al híbrido básico por separado.

**Diagnóstico previo que motiva este notebook.** Se midió fuera de muestra cuánto predice la codificación de ingreso al residuo real, en función del encogimiento:

| ALPHA | 5 | 10 | 25 | 100 (notebook 5) | 400 |
|---|---|---|---|---|---|
| corr con el residuo | **0,1393** | 0,1376 | 0,1334 | 0,1242 | 0,1133 |

La mediana de filas por valor de ingreso es **8**, así que `ALPHA=100` deja sobrevivir apenas el **7,4 %** del efecto crudo. El parámetro que gobierna todo el mecanismo quedó unas veinte veces por encima de su óptimo, y es lo más barato de corregir.

**Hipótesis**
1. **Calibrar el encogimiento del mapa exacto** (`ALPHA` 100 → 5). El mapa redondeado a 100 conserva `ALPHA=100`: sus grupos tienen ~565 filas y ahí el encogimiento sí es razonable.
2. **Correcciones aproximadas en log-odds** (paso de Newton regularizado). El mismo residuo de probabilidad no significa lo mismo cerca de 0,005 que de 0,5, y el denominador `H=Σp(1−p)` pondera por información real en vez de por conteo.
3. **Ingreso condicionado por ambiente**, sólo a granularidad de 1.000.
4. **Retirar movilidad y perfil** de la combinación ganadora: la ablación conjunta aún no se midió.
5. **Profundidad 4 vs 6 vs 9** con idénticas features.

**Calibración de λ.** El mismo barrido aplicado a la fórmula de Newton da su óptimo en λ ≈ 0,5-1, no en 20:

| λ | 0,5 | 1 | 5 | 20 | 50 |
|---|---|---|---|---|---|
| corr con el residuo | **0,1271** | 0,1269 | 0,1222 | 0,1141 | 0,1081 |

Con curvatura mediana `H = 0,63` por grupo, λ=20 conserva el **3,1 %** del paso de Newton — equivale a `ALPHA ≈ 139`, o sea más encogimiento todavía que el 100 ya sobrepasado. λ no es un reescalado (los árboles son invariantes a eso): cambia el orden relativo entre grupos con distinta curvatura, y a 20 los ordena peor. Se fija en **0,7**, el equivalente en unidades de curvatura al `ALPHA=5` óptimo.

**Descartado sin ejecutar, por medición previa:**
- **Distancia diaria.** Residualizando contra el OOF del notebook 5, su ratio de varianza por valor exacto da 0,68 — el modelo ya la explica mejor que el ruido de muestreo, con 99,99 % de cobertura en test. Nada que codificar.
- **`ingreso × ambiente` e `ingreso × subsidio`.** 43.228 y 21.519 grupos, con 29 % y 34 % de grupos de dos filas o menos. Se colapsarían al padre y sólo agregarían columnas redundantes. `ingreso_1000 × ambiente` sí queda: 715 grupos con mediana de 677 filas.

Se mantienen splines, folds y parámetros del notebook 5 salvo los cambios explícitos. No se presupone mejora. Una diferencia favorable entre leaderboard y CV tampoco descarta sobreajuste.

Ejecutar primero con **SMOKE=True**. El modo completo genera OOF y CSVs locales para seis variantes. No envía nada a Kaggle. El notebook es autónomo: las funciones conservadas se copiaron al generarlo; no ejecuta el notebook anterior.


In [1]:
from pathlib import Path
import os
import json
import hashlib
import time
import platform
import importlib.metadata as metadata

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier


SEED=42
SMOKE=os.environ.get("EV_SMOKE","0")=="1"
N_FOLDS=2 if SMOKE else 5
INNER_FOLDS=2 if SMOKE else 3
N_TREES=30 if SMOKE else 600
N_JOBS=4
# Encogimiento del mapa de residuos, por granularidad. El mapa exacto tiene grupos de
# ~8 filas y ALPHA=100 le dejaba el 7,4% del efecto; el redondeado a 100 tiene ~565
# filas por grupo, donde 100 si es razonable. Se calculan ambos alphas del mapa exacto
# para poder aislar el efecto del cambio en las ablaciones.
ALPHA_GRID={"exact":[100.0,5.0],"100":[100.0]}
# Unidades de curvatura, no de filas: H mediana por grupo es 0,63. LAMBDA=0,7 es el
# equivalente al ALPHA=5 optimo (lambda = alpha * p(1-p), con p(1-p) medio = 0,144).
LAMBDA_NEWTON=0.7
MAX_CORRECTION=2.0
TARGET="Will_Buy_EV"
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents]
           if (p/"data/train.csv").exists()),None)
if ROOT is None: raise FileNotFoundError("Abrir desde notebook/ o raíz del episodio.")
OUT=ROOT/"models"/("6_hierarchical_smoke" if SMOKE else "6_hierarchical")
OUT.mkdir(parents=True,exist_ok=True)
VERSIONS={p:metadata.version(p) for p in ["numpy","pandas","scipy","scikit-learn","xgboost"]}
print("SMOKE:",SMOKE,"| versiones:",VERSIONS)
print("ALPHA por granularidad:",ALPHA_GRID,"| LAMBDA_NEWTON:",LAMBDA_NEWTON)


SMOKE: False | versiones: {'numpy': '2.0.2', 'pandas': '2.2.2', 'scipy': '1.14.1', 'scikit-learn': '1.7.2', 'xgboost': '2.1.3'}
ALPHA por granularidad: {'exact': [100.0, 5.0], '100': [100.0]} | LAMBDA_NEWTON: 0.7


In [2]:
train = pd.read_csv(ROOT / "data/train.csv")
test = pd.read_csv(ROOT / "data/test.csv")
sample = pd.read_csv(ROOT / "data/sample_submission.csv")
assert train[TARGET].isin(["Yes", "No"]).all()
assert train["id"].is_unique and test["id"].is_unique
assert test["id"].equals(sample["id"])
original_rows = np.arange(len(train))

if SMOKE:
    original_rows, _ = train_test_split(
        original_rows, train_size=min(12000, len(train) - 2),
        stratify=train[TARGET], random_state=SEED)
    original_rows = np.sort(original_rows)
    train = train.iloc[original_rows].reset_index(drop=True)
    test = test.iloc[:min(1500, len(test))].copy().reset_index(drop=True)
    sample = sample.iloc[:len(test)].copy().reset_index(drop=True)

y = train[TARGET].eq("Yes").to_numpy(dtype=np.int8)
X = train.drop(columns=["id", TARGET])
XT = test.drop(columns="id")
assert set(X.columns) == set(XT.columns)
XT = XT[X.columns]
CAT = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
NUM = [c for c in X if c not in CAT]
print(f"Train: {X.shape}; test: {XT.shape}; positivos: {y.mean():.4%}")
print("Repetición de ingresos (diagnóstico, sin usar target):")
for label, values in [("exacto", X["Annual_Income_USD"]),
                       ("redondeado a 100", X["Annual_Income_USD"].round(-2))]:
    counts = values.value_counts()
    print(label, "valores únicos:", len(counts),
          "| proporción de filas con valor repetido:", values.duplicated(False).mean())


Train: (668665, 13); test: (286571, 13); positivos: 17.4645%
Repetición de ingresos (diagnóstico, sin usar target):
exacto valores únicos: 13214 | proporción de filas con valor repetido: 0.9944142433056912
redondeado a 100 valores únicos: 1184 | proporción de filas con valor repetido: 0.9999446658640725


## 1. Referencia e integridad

Se verifican hashes, IDs, etiquetas y folds del notebook 5. Su OOF solo sirve para evaluar, nunca como feature. En SMOKE no se compara con el baseline completo.

El diagnóstico de cobertura usa únicamente X; la construcción de features posteriores se hace dentro de entrenamiento.

In [3]:

def file_hash(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
input_hashes={n:file_hash(ROOT/"data"/n) for n in ["train.csv","test.csv","sample_submission.csv"]}
baseline=None
if not SMOKE:
    old_config=json.loads((ROOT/"models/5_hybrid/config.json").read_text())
    assert old_config["input_sha256"]==input_hashes,"Dataset distinto del notebook 5"
    with np.load(ROOT/"models/5_hybrid/predictions.npz") as z:
        baseline={k:z[k].copy() for k in ["train_ids","test_ids","y","fold_ids","hybrid_all_oof","hybrid_all_test"]}
    assert np.array_equal(baseline["train_ids"],train["id"])
    assert np.array_equal(baseline["test_ids"],test["id"])
    assert np.array_equal(baseline["y"],y)
    print("Referencia OOF:",roc_auc_score(y,baseline["hybrid_all_oof"]))
    print("Versiones baseline:",old_config["versions"])
coverage=[]
for col in ["Annual_Income_USD","Daily_Commute_km"]:
    counts=X[col].value_counts()
    coverage.append({"feature":col,"unique_train":len(counts),
                     "repeated_rows":X[col].duplicated(False).mean(),
                     "test_seen":XT[col].isin(counts.index).mean(),
                     "test_support_ge20":XT[col].map(counts).fillna(0).ge(20).mean()})
print(pd.DataFrame(coverage).to_string(index=False))


Referencia OOF: 0.9451585586665369
Versiones baseline: {'numpy': '2.0.2', 'pandas': '2.2.2', 'scipy': '1.14.1', 'scikit-learn': '1.7.2', 'xgboost': '2.1.3'}
          feature  unique_train  repeated_rows  test_seen  test_support_ge20
Annual_Income_USD         13214       0.994414   0.994225           0.962282
 Daily_Commute_km           805       0.999937   0.999885           0.998705


## 2. Componentes conservados

Se conservan el modelo aditivo y los mapas originales del notebook 5. Cada fila de entrenamiento recibe features de un ajuste interno que excluye su etiqueta. Validación/test reciben el promedio de modelos internos.

Persiste la asimetría de ruido entre una codificación excluida y el promedio de varias: este experimento no pretende resolverla. Las predicciones usadas para estimar residuos de referencia son in-sample dentro del ajuste interno, por lo que pueden contraer esos residuos, pero no incluyen las etiquetas de las filas que se codifican.

In [4]:
SMOOTH = ["Annual_Income_USD", "Daily_Commute_km"]
ADDITIVE_CAT = CAT + ["env_category", "home_city"]
LINEAR = [c for c in NUM if c not in SMOOTH + ["Environmental_Concern_Level"]]

def additive_frame(df):
    d = df.copy()
    d["env_category"] = d["Environmental_Concern_Level"].astype(str)
    d["home_city"] = (d["Home_Charging_Possible"].astype(str)
                      + "|" + d["City_Type"].astype(str))
    for c in ADDITIVE_CAT:
        d[c] = d[c].fillna("__MISSING__").astype(str)
    return d

def make_additive():
    pre = ColumnTransformer([
        ("smooth", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("spline", SplineTransformer(n_knots=6, degree=3,
                knots="quantile", include_bias=False, extrapolation="linear")),
        ]), SMOOTH),
        ("linear", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), LINEAR),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ADDITIVE_CAT),
    ], sparse_threshold=1.0)
    return Pipeline([
        ("pre", pre),
        ("scale", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(C=0.1, max_iter=600,
                                      solver="lbfgs", random_state=SEED)),
    ])


In [5]:
def deterministic_frame(df):
    d = df.copy()
    income = d["Annual_Income_USD"]
    d["income_band_5k"] = np.floor(income / 5000)
    d["income_position_5k"] = income / 5000 - d["income_band_5k"]
    d["income_distance_1k"] = np.abs(income / 1000 - np.round(income / 1000))
    d["income_round_100"] = np.round(income / 100) * 100

    no_home = d["Home_Charging_Possible"].eq("No").astype(float)
    h = d["Charging_Stations_Near_Home"]
    w = d["Charging_Stations_Near_Work"]
    commute = d["Daily_Commute_km"]
    d["public_commute"] = no_home * commute
    d["charging_min"] = np.minimum(h, w)
    d["charging_max"] = np.maximum(h, w)
    d["charging_imbalance"] = np.abs(h - w)
    d["no_stations_home"] = h.eq(0).astype(float)
    d["no_stations_work"] = w.eq(0).astype(float)
    d["public_friction"] = no_home * commute / (1 + h + w)
    d["anxiety_without_home"] = (
        d["Range_Anxiety_Level"].map({"Low": 0, "Medium": 1, "High": 2}) * no_home)
    d["adoption_profile"] = (
        d["Environmental_Concern_Level"].astype(str) + "|"
        + d["Subsidy_Available"].astype(str) + "|"
        + d["Range_Anxiety_Level"].astype(str))
    d["home_city"] = (
        d["Home_Charging_Possible"].astype(str) + "|" + d["City_Type"].astype(str))
    return d.replace([np.inf, -np.inf], np.nan)

INCOME_COLS = ["income_band_5k", "income_position_5k",
               "income_distance_1k", "income_round_100",
               "income_freq_exact", "income_freq_100"]
# Movilidad y perfil se siguen calculando pero ninguna variante los usa: el notebook 5
# los midio en -0,00004 cada uno. La variante core mide retirarlos en conjunto.
MOBILITY_COLS = ["public_commute", "charging_min", "charging_max",
                 "charging_imbalance", "no_stations_home", "no_stations_work",
                 "public_friction", "anxiety_without_home"]
# PROFILE_COLS se necesita para fijar el dtype categorico, aunque no entre al modelo.
PROFILE_COLS = ["adoption_profile", "home_city"]


In [6]:
def income_keys(df):
    return {
        "exact": df["Annual_Income_USD"],
        "100": np.round(df["Annual_Income_USD"] / 100) * 100,
    }

def residual_col(name, alpha):
    return f"income_residual_{name}_a{int(alpha)}"

RESIDUAL_ALL = [residual_col(n, a) for n, alphas in ALPHA_GRID.items() for a in alphas]
LEARNED = ["income_freq_exact", "income_freq_100"] + RESIDUAL_ALL
# Los dos juegos que se comparan: el del notebook 5 y el calibrado.
RESIDUAL_A100 = [residual_col("exact", 100.0), residual_col("100", 100.0)]
RESIDUAL_A5   = [residual_col("exact", 5.0),   residual_col("100", 100.0)]

def fit_income_maps(df, residual):
    """Guarda sumas y conteos crudos; el encogimiento se aplica al transformar,
    asi el mismo ajuste sirve para varios alphas sin recalcular el groupby."""
    result = {}
    for name, key in income_keys(df).items():
        tab = pd.DataFrame({"key": key.to_numpy(), "residual": residual})
        agg = tab.groupby("key")["residual"].agg(["sum", "count"])
        result[name] = {"freq": agg["count"] / len(df),
                        "sum": agg["sum"], "count": agg["count"]}
    return result

def apply_income_maps(df, maps):
    out = pd.DataFrame(index=df.index)
    for name, key in income_keys(df).items():
        out[f"income_freq_{name}"] = key.map(maps[name]["freq"]).fillna(0)
        for alpha in ALPHA_GRID[name]:
            shrunk = maps[name]["sum"] / (maps[name]["count"] + alpha)
            out[residual_col(name, alpha)] = key.map(shrunk).fillna(0)
    return out[LEARNED]


## 3. Corrección Newton y contracción jerárquica

Para un grupo g, usando probabilidades del aditivo:
\[
G_g=\sum(y_i-p_i),\quad H_g=\sum p_i(1-p_i),\quad
d_g=\operatorname{clip}(G_g/(H_g+\lambda),-2,2).
\]
Es un paso Newton regularizado aproximado para un desplazamiento de log-odds, no una probabilidad exacta. Se incorpora como feature; XGBoost decide cuánto usarlo.

Para un grupo hijo c:
\[
d_{g,c}=\operatorname{clip}((G_{g,c}+\lambda d_g)/(H_{g,c}+\lambda),-2,2).
\]
Grupos pequeños se contraen hacia el ingreso padre; hijos desconocidos heredan el padre; padres desconocidos reciben cero. Se añaden soporte e información relativa H/(H+lambda).

**Mapas conservados:** ingreso exacto, redondeado a 100, redondeado a 1.000, y `ingreso_1000 × ambiente` contraído hacia su padre. Los mapas de `ingreso × ambiente`, `ingreso × subsidio`, distancia y `distancia × ansiedad` se retiraron por medición previa (ver encabezado): los dos primeros por fragmentación, los dos últimos porque la distancia ya no tiene varianza residual por valor. Las interacciones describen correcciones del modelo, no efectos causales demostrados.

**Sobre el clip.** Con λ=20 nunca se activaba. Con λ=0,7 alcanza a un ~0,3 % de las filas, que es el rol que se espera de un guardarraíl: acotar los grupos más extremos sin tocar el resto.


In [7]:

def key_frame(df):
    return pd.DataFrame({
        "income":df["Annual_Income_USD"].to_numpy(),
        "income100":(np.round(df["Annual_Income_USD"]/100)*100).to_numpy(),
        "income1000":(np.round(df["Annual_Income_USD"]/1000)*1000).to_numpy(),
        "env":df["Environmental_Concern_Level"].to_numpy()})
# Los padres deben preceder a sus hijos: apply_newton_maps usa el valor ya calculado
# del padre como fallback cuando el hijo no aparece en el mapa.
SPECS=[
    ("inc",["income"],None),
    ("inc100",["income100"],None),
    ("inc1000",["income1000"],None),
    ("inc1000_env",["income1000","env"],"inc1000")]
def lookup_series(series,keys,cols):
    idx=(pd.Index(keys[cols[0]].to_numpy(),name=cols[0]) if len(cols)==1
         else pd.MultiIndex.from_frame(keys[cols]))
    return series.reindex(idx).to_numpy()
def fit_newton_maps(df,target,prob):
    keys=key_frame(df)
    keys["grad"]=target-prob
    keys["hess"]=prob*(1-prob)
    maps={}
    for name,cols,parent in SPECS:
        agg=keys.groupby(cols,dropna=False,observed=True).agg(
            grad=("grad","sum"),hess=("hess","sum"),count=("grad","size"))
        prior=np.zeros(len(agg))
        if parent is not None:
            parent_cols,parent_tab=maps[parent]
            prior=np.nan_to_num(lookup_series(parent_tab["delta"],agg.reset_index(),parent_cols))
        agg["delta"]=np.clip((agg["grad"]+LAMBDA_NEWTON*prior)/(agg["hess"]+LAMBDA_NEWTON),
                             -MAX_CORRECTION,MAX_CORRECTION)
        maps[name]=(cols,agg)
    return maps
def apply_newton_maps(df,maps):
    keys=key_frame(df)
    out=pd.DataFrame(index=np.arange(len(df)))
    for name,cols,parent in SPECS:
        tab=maps[name][1]
        delta=lookup_series(tab["delta"],keys,cols)
        fallback=out[f"newton_{parent}"].to_numpy() if parent else np.zeros(len(df))
        out[f"newton_{name}"]=np.where(np.isnan(delta),fallback,delta)
        hess=np.nan_to_num(lookup_series(tab["hess"],keys,cols))
        count=np.nan_to_num(lookup_series(tab["count"],keys,cols))
        out[f"reliability_{name}"]=hess/(hess+LAMBDA_NEWTON)
        out[f"log_count_{name}"]=np.log1p(count)
    return out
def map_columns(groups):
    return [f"{prefix}_{g}" for g in groups for prefix in ["newton","reliability","log_count"]]
NEWTON_COLS=map_columns(["inc","inc100","inc1000"])
CONTEXT_COLS=map_columns(["inc1000_env"])
NEW_COLS=NEWTON_COLS+CONTEXT_COLS
CORE_A100=INCOME_COLS+RESIDUAL_A100   # receta del notebook 5, sin movilidad ni perfil
CORE_A5=INCOME_COLS+RESIDUAL_A5       # idem, con el encogimiento calibrado
# Cada variante cambia una sola cosa respecto de la anterior.
VARIANTS={
    "core_a100":(CORE_A100,4),          # control: mide retirar movilidad y perfil
    "core_a5":(CORE_A5,4),              # aisla el efecto de ALPHA 100 -> 5
    "newton":(CORE_A5+NEWTON_COLS,4),   # suma la correccion en log-odds
    "newton_context":(CORE_A5+NEW_COLS,4),   # suma ingreso_1000 x ambiente
    "newton_context_d6":(CORE_A5+NEW_COLS,6),
    "newton_context_d9":(CORE_A5+NEW_COLS,9)}
for name,(extra,depth) in VARIANTS.items():
    print(f"  {name:20s} depth={depth}  features extra={len(extra)}")


  core_a100            depth=4  features extra=8
  core_a5              depth=4  features extra=8
  newton               depth=4  features extra=17
  newton_context       depth=4  features extra=20
  newton_context_d6    depth=6  features extra=20
  newton_context_d9    depth=9  features extra=20


## 4. Comprobaciones de fallback y transformación dentro de folds

La prueba controlada verifica hijos nuevos con padre conocido y padres desconocidos. El bucle interno comprueba cobertura OOF una vez por fila y valores finitos.

Se mantienen 600 árboles fijados antes de evaluar, sin early stopping exterior. Cada variante recibe idénticos márgenes y particiones.

In [8]:

# Fila 0: hijo nuevo (ambiente no visto) con padre conocido -> debe heredar al padre.
# Fila 1: ingreso nunca visto -> correccion cero y confiabilidad cero.
toy=pd.DataFrame({"Annual_Income_USD":[50000.,50000.,60000.],
    "Environmental_Concern_Level":[1.,1.,2.]})
maps=fit_newton_maps(toy,np.array([1,0,1]),np.array([.2,.3,.4]))
probe=toy.iloc[[0,1]].copy().reset_index(drop=True)
probe.loc[0,"Environmental_Concern_Level"]=99.
probe.loc[1,"Annual_Income_USD"]=999999.
check=apply_newton_maps(probe,maps)
assert np.isfinite(check.to_numpy()).all()
assert check.loc[0,"newton_inc1000_env"]==check.loc[0,"newton_inc1000"]
assert check.loc[1,"newton_inc"]==0 and check.loc[1,"reliability_inc"]==0
assert check.loc[1,"newton_inc1000_env"]==0
print("Fallback OK")
def stage_one_extended(a,ya,b,c,seed):
    a,b,c=[d.reset_index(drop=True) for d in [a,b,c]]
    aa,bb,cc=[additive_frame(d) for d in [a,b,c]]
    mt=np.full(len(a),np.nan)
    mv,ms=np.zeros(len(b)),np.zeros(len(c))
    cols=LEARNED+NEW_COLS
    lt=pd.DataFrame(np.nan,index=a.index,columns=cols)
    lv=pd.DataFrame(0.,index=b.index,columns=cols)
    ls=pd.DataFrame(0.,index=c.index,columns=cols)
    seen=np.zeros(len(a),dtype=int)
    inner=StratifiedKFold(INNER_FOLDS,shuffle=True,random_state=seed)
    for fit_idx,hold_idx in inner.split(a,ya):
        assert not np.intersect1d(fit_idx,hold_idx).size
        model=make_additive()
        model.fit(aa.iloc[fit_idx],ya[fit_idx])
        mt[hold_idx]=model.decision_function(aa.iloc[hold_idx])
        mv+=model.decision_function(bb)/INNER_FOLDS
        ms+=model.decision_function(cc)/INNER_FOLDS
        prob=model.predict_proba(aa.iloc[fit_idx])[:,1]
        old_maps=fit_income_maps(a.iloc[fit_idx],ya[fit_idx]-prob)
        new_maps=fit_newton_maps(a.iloc[fit_idx],ya[fit_idx],prob)
        def transform(raw):
            left=apply_income_maps(raw,old_maps).reset_index(drop=True)
            right=apply_newton_maps(raw,new_maps)
            return pd.concat([left,right],axis=1)[cols].to_numpy()
        lt.iloc[hold_idx]=transform(a.iloc[hold_idx])
        lv+=transform(b)/INNER_FOLDS
        ls+=transform(c)/INNER_FOLDS
        seen[hold_idx]+=1
    assert np.all(seen==1)
    for z in [mt,mv,ms,lt,lv,ls]: assert np.isfinite(np.asarray(z)).all()
    frames=[]
    for raw,learned in [(a,lt),(b,lv),(c,ls)]:
        f=deterministic_frame(raw)
        f[cols]=learned.to_numpy()
        frames.append(f)
    for col in CAT+PROFILE_COLS:
        dtype=pd.CategoricalDtype(sorted(frames[0][col].dropna().astype(str).unique()))
        for f in frames: f[col]=f[col].astype(dtype)
    return mt,mv,ms,frames


Fallback OK


## 5. Validación exterior y ablaciones

Seis variantes × cinco folds, compartiendo la etapa aditiva por fold. Cada una cambia **una sola cosa** respecto de la anterior, para que cada delta sea atribuible:

| Variante | Qué mide |
|---|---|
| `core_a100` | Retirar movilidad y perfil, con la receta exacta del notebook 5. Comparable contra su `hybrid_all`. |
| `core_a5` | El efecto de calibrar `ALPHA` de 100 a 5, y nada más. |
| `newton` | Lo que agrega la corrección en log-odds sobre el encoding ya calibrado. |
| `newton_context` | Lo que agrega `ingreso_1000 × ambiente`. |
| `newton_context_d6` / `_d9` | Profundidad, con features idénticas. |

Separar `core_a100` de `core_a5` es lo que permite no confundir la calibración con el mecanismo nuevo: si se cambiaran las dos cosas a la vez, una mejora conjunta no diría cuál de las dos la produjo.

Se guardan checkpoints por fold con IDs. Reiniciar vuelve a calcular los folds; no se reutilizan checkpoints de otra configuración.


In [9]:

PARAMS=dict(objective="binary:logistic",eval_metric="auc",n_estimators=N_TREES,
    learning_rate=0.05,min_child_weight=20,subsample=0.9,colsample_bytree=1.0,
    reg_lambda=10.0,reg_alpha=0.1,tree_method="hist",enable_categorical=True,
    max_bin=256,n_jobs=N_JOBS,random_state=SEED)
oof={n:np.full(len(X),np.nan,dtype=np.float32) for n in VARIANTS}
test_pred={n:np.zeros(len(XT)) for n in VARIANTS}
fold_ids=np.full(len(X),-1,dtype=np.int8)
records=[]
outer=StratifiedKFold(N_FOLDS,shuffle=True,random_state=SEED)
started=time.perf_counter()
for fold,(tr,va) in enumerate(outer.split(X,y),1):
    fold_ids[va]=fold
    if baseline is not None:
        assert np.array_equal(np.flatnonzero(baseline["fold_ids"]==fold),va)
    print(f"Fold {fold}/{N_FOLDS}: aditivo y mapas...",flush=True)
    mt,mv,ms,(fa,fb,fc)=stage_one_extended(X.iloc[tr],y[tr],X.iloc[va],XT,SEED+fold)
    checkpoint={"valid_indices":va,"valid_ids":train["id"].to_numpy()[va]}
    for name,(extra,depth) in VARIANTS.items():
        cols=list(X.columns)+extra
        assert len(cols)==len(set(cols))
        model=XGBClassifier(**PARAMS,max_depth=depth)
        model.fit(fa[cols],y[tr],base_margin=mt,verbose=False)
        pv=model.predict_proba(fb[cols],base_margin=mv)[:,1]
        pt=model.predict_proba(fc[cols],base_margin=ms)[:,1]
        assert np.isfinite(pv).all() and np.isfinite(pt).all()
        assert ((pv>=0)&(pv<=1)).all() and ((pt>=0)&(pt<=1)).all()
        oof[name][va]=pv
        test_pred[name]+=pt/N_FOLDS
        score=roc_auc_score(y[va],pv)
        records.append({"model":name,"fold":fold,"auc":score})
        checkpoint[name+"_valid"]=pv
        checkpoint[name+"_test"]=pt
        print(f"  {name:22s}: {score:.6f}",flush=True)
    np.savez_compressed(OUT/f"fold_{fold}.npz",**checkpoint)
assert (fold_ids>0).all() and all(np.isfinite(p).all() for p in oof.values())
print(f"Tiempo: {(time.perf_counter()-started)/60:.1f} min")


Fold 1/5: aditivo y mapas...
  core_a100             : 0.944208
  core_a5               : 0.944237
  newton                : 0.944495
  newton_context        : 0.944499
  newton_context_d6     : 0.944442
  newton_context_d9     : 0.943960
Fold 2/5: aditivo y mapas...
  core_a100             : 0.944914
  core_a5               : 0.944950
  newton                : 0.945132
  newton_context        : 0.945130
  newton_context_d6     : 0.945075
  newton_context_d9     : 0.944548
Fold 3/5: aditivo y mapas...
  core_a100             : 0.946131
  core_a5               : 0.946183
  newton                : 0.946439
  newton_context        : 0.946427
  newton_context_d6     : 0.946369
  newton_context_d9     : 0.945731
Fold 4/5: aditivo y mapas...
  core_a100             : 0.945462
  core_a5               : 0.945488
  newton                : 0.945737
  newton_context        : 0.945693
  newton_context_d6     : 0.945547
  newton_context_d9     : 0.944916
Fold 5/5: aditivo y mapas...
  core_a100    

## 6. Comparación pareada y mezcla fija

Se separan AUC OOF global y media por fold. No se compara la mejora con la dispersión de scores entre folds como prueba de significancia — ése es el estadístico equivocado, porque los folds son los mismos para todas las variantes y esa varianza se cancela al restar. Se reporta el **desvío de la diferencia pareada**, que es lo que corresponde. Aun así, los entrenamientos se solapan y se están evaluando varias hipótesis: las diferencias siguen siendo exploratorias.

Se prueba además una mezcla predefinida 50/50 entre el ganador del notebook 5 y `newton_context`, sin buscar pesos sobre el OOF.


In [10]:

BEST="newton_context"   # fijado de antemano, no elegido por el mayor OOF
if baseline is not None:
    oof["previous_hybrid_all"]=baseline["hybrid_all_oof"]
    test_pred["previous_hybrid_all"]=baseline["hybrid_all_test"]
    oof["blend_50"]=.5*oof[BEST]+.5*baseline["hybrid_all_oof"]
    test_pred["blend_50"]=.5*test_pred[BEST]+.5*baseline["hybrid_all_test"]
    for name in ["previous_hybrid_all","blend_50"]:
        for f in range(1,N_FOLDS+1):
            mask=fold_ids==f
            records.append({"model":name,"fold":f,"auc":roc_auc_score(y[mask],oof[name][mask])})
fold_metrics=pd.DataFrame(records)
paired=fold_metrics.pivot(index="fold",columns="model",values="auc")
summary=pd.DataFrame({n:{"auc_oof":roc_auc_score(y,p),
    "mean_fold":paired[n].mean(),"std_fold":paired[n].std(ddof=1)}
    for n,p in oof.items()}).T
if baseline is not None:
    summary["delta_oof_vs_5"]=summary["auc_oof"]-summary.loc["previous_hybrid_all","auc_oof"]
    delta=paired.subtract(paired["previous_hybrid_all"],axis=0)
    summary["folds_won_vs_5"]=(delta>0).sum()
    print("Diferencias por fold contra notebook 5:")
    print(delta.round(6).to_string())
print(summary.sort_values("auc_oof",ascending=False).round(6).to_string())

print("\nAblaciones (cada una aisla un solo cambio):")
ABLACIONES=[("core_a5","core_a100","calibrar ALPHA 100 -> 5"),
            ("newton","core_a5","correccion en log-odds"),
            ("newton_context","newton","ingreso_1000 x ambiente"),
            ("newton_context_d6","newton_context","profundidad 4 -> 6"),
            ("newton_context_d9","newton_context","profundidad 4 -> 9")]
if baseline is not None:
    ABLACIONES.insert(0,("core_a100","previous_hybrid_all","retirar movilidad y perfil"))
for new,control,que in ABLACIONES:
    d=paired[new]-paired[control]
    print(f"  {que:32s} {d.mean():+.6f}  | folds ganados: {int((d>0).sum())}/{N_FOLDS}"
          f"  | sd pareado: {d.std(ddof=1):.6f}")
summary.to_csv(OUT/"metrics.csv")
fold_metrics.to_csv(OUT/"fold_metrics.csv",index=False)


Diferencias por fold contra notebook 5:
model  blend_50  core_a100   core_a5    newton  newton_context  newton_context_d6  newton_context_d9  previous_hybrid_all
fold                                                                                                                     
1      0.000261   0.000032  0.000061  0.000319        0.000323           0.000266          -0.000216                  0.0
2      0.000199  -0.000012  0.000024  0.000206        0.000204           0.000149          -0.000378                  0.0
3      0.000260   0.000018  0.000071  0.000327        0.000315           0.000257          -0.000381                  0.0
4      0.000239   0.000030  0.000056  0.000305        0.000260           0.000115          -0.000516                  0.0
5      0.000285   0.000060  0.000068  0.000385        0.000370           0.000328          -0.000253                  0.0
                      auc_oof  mean_fold  std_fold  delta_oof_vs_5  folds_won_vs_5
newton               0.

## 7. Diagnóstico por segmento y soporte

Soporte del ingreso calculado sobre entrenamiento exterior, excluyendo validación. Se omite el AUC de grupos con menos de 20 positivos o negativos: ansiedad alta tenía apenas tres positivos en todo el dataset. El AUC por segmento no suma el global, que también contiene pares entre segmentos.

In [11]:

support=np.zeros(len(X))
for f in range(1,N_FOLDS+1):
    tr_mask,va_mask=fold_ids!=f,fold_ids==f
    counts=X.loc[tr_mask,"Annual_Income_USD"].value_counts()
    support[va_mask]=X.loc[va_mask,"Annual_Income_USD"].map(counts).fillna(0)
buckets=pd.cut(support,[-1,0,10,50,np.inf],labels=["unseen","1-10","11-50",">50"])
segment_records=[]
for column,values in [("Subsidy",X["Subsidy_Available"]),
                      ("Anxiety",X["Range_Anxiety_Level"]),("Income_support",pd.Series(buckets))]:
    for value in values.unique():
        mask=np.asarray(values==value)
        pos,neg=int(y[mask].sum()),int(mask.sum()-y[mask].sum())
        for name in oof:
            auc=roc_auc_score(y[mask],oof[name][mask]) if min(pos,neg)>=20 else np.nan
            segment_records.append({"segment":column,"value":str(value),"model":name,
                "rows":int(mask.sum()),"positives":pos,"auc":auc})
segments=pd.DataFrame(segment_records)
segments.to_csv(OUT/"segments.csv",index=False)
# El bucket de soporte es el diagnostico clave: si calibrar ALPHA sirve, la mejora
# deberia concentrarse en los grupos de poco soporte, que son los que estaban
# sobre-encogidos.
mostrar=[n for n in [BEST,"core_a5","core_a100","previous_hybrid_all"] if n in oof]
print(segments[segments.model.isin(mostrar)].round(6).to_string(index=False))


       segment  value               model   rows  positives      auc
       Subsidy     No           core_a100 248756       1432 0.885424
       Subsidy     No             core_a5 248756       1432 0.885636
       Subsidy     No      newton_context 248756       1432 0.885615
       Subsidy     No previous_hybrid_all 248756       1432 0.886040
       Subsidy    Yes           core_a100 419909     115347 0.909701
       Subsidy    Yes             core_a5 419909     115347 0.909739
       Subsidy    Yes      newton_context 419909     115347 0.910145
       Subsidy    Yes previous_hybrid_all 419909     115347 0.909639
       Anxiety    Low           core_a100 603972     114167 0.942593
       Anxiety    Low             core_a5 603972     114167 0.942626
       Anxiety    Low      newton_context 603972     114167 0.942870
       Anxiety    Low previous_hybrid_all 603972     114167 0.942576
       Anxiety Medium           core_a100  62499       2609 0.937079
       Anxiety Medium             

## 8. Artefactos y submissions

Se guardan todos los OOF/test con IDs, folds, configuración, versiones y hashes. En completo se escribe un CSV por variante y la mezcla fija, con nombres explícitos. No se selecciona ni envía automáticamente un ganador; revisar metrics.csv.

Elegir la mejor variante mirando esta CV introduce sesgo de selección. Confirmar ganancias pequeñas con evaluación adicional. Otra semilla ayuda a medir estabilidad pero no crea datos independientes.

In [12]:

payload={"train_ids":train["id"].to_numpy(),"test_ids":test["id"].to_numpy(),
         "fold_ids":fold_ids,"y":y,"original_rows":original_rows}
for name in oof:
    payload[name+"_oof"]=oof[name]
    payload[name+"_test"]=test_pred[name]
np.savez_compressed(OUT/"predictions.npz",**payload)
config={"smoke":SMOKE,"seed":SEED,"outer_folds":N_FOLDS,"inner_folds":INNER_FOLDS,
        "alpha_grid":ALPHA_GRID,"lambda_newton":LAMBDA_NEWTON,"max_correction":MAX_CORRECTION,
        "variants":{k:[v[0],v[1]] for k,v in VARIANTS.items()},
        "map_specs":[[n,c,p] for n,c,p in SPECS],"xgb_params":PARAMS,
        "additive":{"n_knots":6,"degree":3,"C":0.1},"versions":VERSIONS,"input_sha256":input_hashes}
(OUT/"config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")
if SMOKE:
    print("SMOKE OK; no submissions. Scores no comparables con CV completa.")
else:
    destination=ROOT/"submissions"
    destination.mkdir(exist_ok=True)
    for name in [*VARIANTS,"blend_50"]:
        sub=pd.DataFrame({"id":test["id"],TARGET:test_pred[name]})
        assert list(sub.columns)==list(sample.columns) and sub["id"].equals(sample["id"])
        assert sub[TARGET].notna().all() and sub[TARGET].between(0,1).all()
        sub.to_csv(destination/f"6_{name}_submission.csv",index=False)
    print("Artefactos:",OUT)
    print("CSVs:",destination)
    print("Revisar metrics.csv antes de elegir un envío.")


Artefactos: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 9 - Predicting Electric Vehicle Purchases\models\6_hierarchical
CSVs: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 9 - Predicting Electric Vehicle Purchases\submissions
Revisar metrics.csv antes de elegir un envío.


## Conclusiones pendientes de ejecución completa

Cada ablación contesta una pregunta y sólo una:

- **`core_a100` vs notebook 5** — ¿retirar movilidad y perfil cambia algo? Esperado ≈ 0,0000: el notebook 5 los midió en −0,00004 cada uno.
- **`core_a5` vs `core_a100`** — ¿la calibración de `ALPHA` se traduce en AUC? Es la hipótesis principal. El barrido daba +12 % de correlación con el residuo; cuánto de eso llega a la métrica es exactamente lo que no se sabe.
- **`newton` vs `core_a5`** — ¿la corrección en log-odds aporta sobre un encoding ya bien calibrado, o las dos cosas estaban capturando lo mismo?
- **`newton_context` vs `newton`** — ¿condicionar por ambiente a granularidad de 1.000 agrega señal?
- **`_d6` / `_d9` vs `newton_context`** — ¿la profundidad 4 heredada del notebook 5 estaba limitando?

Si `core_a5` explica casi toda la mejora y `newton` no agrega nada encima, la conclusión es que el mecanismo del notebook 5 ya era el correcto y sólo estaba mal calibrado. Si pasa lo contrario, la formulación en log-odds es lo que importaba.

La prueba SMOKE valida implementación; no demuestra una mejora en la competencia. Elegir la mejor variante mirando esta CV introduce sesgo de selección: `newton_context` quedó fijado de antemano como candidato para la mezcla.
